# 08 · mcTEC ageing score in Xenium — per-cell ULM, permuted-network control, spatial maps, aged-like fraction per donor, score across the six mcTEC niches

**What this notebook does.** Every mcTEC cell in the seven Xenium sections is scored with the
scRNA-seq **adult-vs-infant differential-expression signature of mcTEC** (log2 fold changes of the
significant genes, restricted to the Xenium panel) using the univariate linear model (ULM) of
`decoupler`, one cell at a time (no spatial pooling, "k = 1"). A positive score means the cell's
expression profile resembles adult mcTEC, a negative score infant mcTEC, and 0 means no resemblance
either way — so every colour scale in this notebook is centred on 0.

**Controls.** (i) The same scoring with **100 permuted networks** (identical weights, target genes
redrawn at random from the scoring panel) gives the null distribution of the score itself and of every
summary statistic; (ii) the **section-mean score is correlated with the donor's chronological age**
(seven sections = seven donors) and compared with the same correlation under the permuted networks.

**Outputs.** Spatial maps of mcTEC coloured by score (centred on 0); the fraction of aged-like mcTEC
per donor; the score across the six mcTEC neighbourhood niches of
`morphoFM/00_mcTEC_niche_VK_k6.ipynb`, drawn as a marginal boxplot in the row order of that
notebook's published heatmaps so it can sit beside them.

**Not here.** The July notebook (`mcTEC_ageDE/Xenium_mcTEC_ageDE_scoring.ipynb`) additionally pooled
counts over k nearest mcTEC, gated on split-half reliability and regressed the score on neighbourhood
composition. None of that is needed for the figures above and it was removed.

**How to re-run.** Kernel `scvi-env3`. Either open it on a Jupyter server, or rebuild + execute inside
an LSF job (nothing runs on the login / VS Code host):

```bash
J=/nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/notebooks/xenium/notebooks_revision/final/_jobs
bsub -q normal -J xen08 -M 24000 -R "select[mem>24000] rusage[mem=24000]" -n 4 \
     -o $J/logs/xen08.%J.log -e $J/logs/xen08.%J.err $J/run_nb.sh 08
```

Runs in ~10 minutes and ~6 GB. The notebook is generated from `_jobs/nb08_mcTEC_ageDE.py`; edit the
builder (or edit the notebook and stop using the builder).

## 1 · Setup — inputs, parameters, colours

In [1]:
import os
import time
import json
import logging
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.sparse as sp
import h5py
import anndata as ad
from anndata.io import sparse_dataset, read_elem
import decoupler as dc
from scipy.stats import spearmanr, kruskal, friedmanchisquare, wilcoxon
from scipy.cluster.hierarchy import linkage, dendrogram
from statsmodels.stats.multitest import multipletests
from itertools import combinations
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager
from matplotlib.colors import to_rgb, to_rgba, to_hex, rgb_to_hsv
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
logging.getLogger('fontTools').setLevel(logging.WARNING)      # PDF font subsetting otherwise logs every glyph
today = datetime.now().strftime('%Y-%m-%d')
T_START = time.time()

REPO = '/nfs/team205/vk8/projects/thymus_ageing_atlas'
SPATIAL = f'{REPO}/Spatial_analyses'
GENERAL = f'{REPO}/General_analysis'
font_manager.fontManager.addfont('/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf')
plt.style.use(f'{GENERAL}/scripts/plotting/thyAgeing.mplstyle')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

# ---- inputs -----------------------------------------------------------------------------------
H5AD_LABELS = f'{SPATIAL}/data/xenium/xenium_proseg/xenium_aging_labels.h5ad'   # .X = log-normalised expression, 3.53 M cells x 5,043 genes
H5AD_PROSEG = f'{SPATIAL}/data/xenium/xenium_proseg/xenium_proseg_aging.h5ad'   # same cells, same order: obsm['spatial'] (um), obs['total_counts']
ANNO_CSV = f'{SPATIAL}/data/xenium/ThyAge_xenium_annos_Fb_upd-clean_2026-09-02.csv'  # cell types; index = h5ad obs_names
PANEL_H5AD = f'{SPATIAL}/data/xenium/pseudobulk_final/pb_cells_taa_l3_upd.h5ad'  # var['shared_panel'] = the 4,993 scoring genes (notebook 02)
DE_CSV = f'{GENERAL}/data/analyses/dea/thyAgeing_dea_taa_l3_adult_vs_infant_ageEffect.csv'  # scRNA-seq pyDESeq2, adult vs infant, per taa_l3 type
NICHE_DIR = f'{SPATIAL}/data/morpho_niches/mctecNiche'                          # outputs of morphoFM/00_mcTEC_niche_VK_k6.ipynb
NICHE_CSV = f'{NICHE_DIR}/mctec_niche_labels_r50_k6.csv'                        # niche label per mcTEC
NICHE_PCT_CSV = f'{NICHE_DIR}/mctec_niche_k6_taa_l2_comb_composition_pct.csv'   # niche x cell type, % of the 50 um neighbourhood
NICHE_ENR_CSV = f'{NICHE_DIR}/mctec_niche_k6_taa_l3_comb_log2enrichment.csv'    # niche x cell type, log2 enrichment (defines the row order)
for p in [H5AD_LABELS, H5AD_PROSEG, ANNO_CSV, PANEL_H5AD, DE_CSV, NICHE_CSV, NICHE_PCT_CSV, NICHE_ENR_CSV]:
    assert os.path.exists(p), p

# ---- outputs ----------------------------------------------------------------------------------
RESULTS = f'{SPATIAL}/results/xenium/final/08_mcTEC_ageDE'
PLOTS = f'{SPATIAL}/plots/xenium/final/08_mcTEC_ageDE'
for d in (RESULTS, PLOTS):
    os.makedirs(d, exist_ok=True)
PREFIX = 'xenium_mcTEC_ageDE'

# ---- parameters -------------------------------------------------------------------------------
CELL_TYPE = 'mcTEC'
CT_KEY = 'taa_l3_upd'                              # annotation column that defines the cells
PADJ_MAX, LFC_MIN, BASEMEAN_MIN = 0.05, 0.5, 10    # signature gene filter on the scRNA-seq DE table
TMIN = 5                                           # decoupler: minimum targets per source
N_PERM = 100                                       # permuted (target-shuffled) networks
SEED = 0
NULL_Q = 0.95                                      # strict aged-like = score above this quantile of all permuted-network scores
MIN_CELLS = 30                                     # smallest group summarised by a median / box

DONOR_ORDER = ['U51', 'U48', 'Z16', 'Z17', 'A67', 'A72', 'A70']            # youngest -> oldest
DONOR_AGE = {'U51': 'epaed', 'U48': 'epaed', 'Z16': 'lpaed', 'Z17': 'lpaed', 'A67': 'adult', 'A72': 'adult', 'A70': 'adult'}
DONOR_AGE_YEARS = {'U51': 0.04, 'U48': 0.5, 'Z16': 8.0, 'Z17': 11.0, 'A67': 20.0, 'A72': 33.0, 'A70': 37.0}
DONOR_AGE_LABEL = {'U51': '2 w', 'U48': '6 m', 'Z16': '8 y', 'Z17': '11 y', 'A67': '20 y', 'A72': '33 y', 'A70': '37 y'}
YOUNG, ADULT = ['U51', 'U48', 'Z16', 'Z17'], ['A67', 'A72', 'A70']
AGE_ORDER = ['epaed', 'lpaed', 'adult']

# project palette (General_analysis/scripts/plotting/thyAgeing_colors.csv / _greys.csv), copied so nothing is imported
THY = {'blue': '#648FFF', 'purple': '#785EF0', 'magenta': '#DC267F', 'orange': '#FE6100', 'yellow': '#FFB000',
       'teal': '#18868F', 'mint': '#738B79', 'pink': '#C169E8', 'grass': '#82CD29', 'forest': '#124420'}
GREY = {'grey1': '#f7f7f7', 'grey2': '#cccccc', 'grey3': '#969696', 'grey4': '#636363', 'grey5': '#252525'}


def tint_palette(base, n, min_tint=0.3, max_tint=1.0):
    """Tints of one colour, white -> base as the tint goes 0 -> 1 (same formula as plotting.utils.get_tint_palette)."""
    r, g, b = to_rgb(base)
    return [to_hex((1 - t + t * r, 1 - t + t * g, 1 - t + t * b)) for t in np.linspace(min_tint, max_tint, n)]


# the study's age-group colours: the magenta tint ramp used in the frequency boxplots and the Fb ageing figures
AGE_COLORS = dict(zip(AGE_ORDER, tint_palette(THY['magenta'], len(AGE_ORDER))))     # epaed light pink -> adult full magenta


def hue(c):
    return rgb_to_hsv(to_rgb(c))[0] * 359


# score colour scale: teal = young-like, white = 0, orange = aged-like (the project's diverging scale)
CMAP_SCORE = sns.diverging_palette(hue(THY['teal']), hue(THY['orange']), s=90, l=60, center='light', as_cmap=True)

# ---- the six mcTEC niches of morphoFM/00_mcTEC_niche_VK_k6.ipynb --------------------------------
NICHE_KEY = 'mctec_niche_r50_k6'
# Row order of that notebook's published heatmaps (mctec_niche_k6_composition_pct_scaled_heatmap.pdf,
# mctec_niche_k6_celltype_distribution.pdf, mctec_niche_k6_morpho_niche_l2_log2oe_pub.pdf): the leaves of
# the average-linkage / correlation-distance tree on the taa_l3 enrichment profiles. Hard-coded so the
# marginal boxplot lines up with those PDFs; re-derived from the saved profiles and checked in section 6.
HEATMAP_ROW_ORDER = ['mcTEC_N0', 'mcTEC_N2', 'mcTEC_N5', 'mcTEC_N4', 'mcTEC_N1', 'mcTEC_N3']
# Colours and short labels of the assembled figure (composition heatmap + spatial enrichment), top row to
# bottom row: C* purple, S teal, FV* yellow, M grass, J* magenta, MV orange. NB the k6 notebook's own
# legends (per-donor bars, section maps) use a different assignment (N3 grass, N4 orange, N0 yellow,
# N5 purple); change this one dict if the figure's colours move.
NICHE_COLORS = {'mcTEC_N0': THY['purple'], 'mcTEC_N2': THY['teal'], 'mcTEC_N5': THY['yellow'],
                'mcTEC_N4': THY['grass'], 'mcTEC_N1': THY['magenta'], 'mcTEC_N3': THY['orange']}
NICHE_SHORT = {'mcTEC_N0': 'C*', 'mcTEC_N2': 'S', 'mcTEC_N5': 'FV*', 'mcTEC_N4': 'M', 'mcTEC_N1': 'J*', 'mcTEC_N3': 'MV'}
BOX_ALPHAS = [(1.0, ''), (0.7, '_alpha07')]        # every boxplot figure is saved once per entry: (face alpha, file suffix)
NICHE_DESC = {'mcTEC_N2': 'young cortex (T_DP-rich)', 'mcTEC_N4': 'medullary (mTEC, B, recirculating T)',
              'mcTEC_N3': 'perimedullary / intermediate', 'mcTEC_N0': 'adult cortical (T_DP, mcTEC, Mac)',
              'mcTEC_N1': 'adult B-cell / vascular / fibrotic', 'mcTEC_N5': 'mcTEC-dense stromal'}
# columns of the published composition heatmap, in its order
HEATMAP_COLS = ['B_cell', 'T_recirc', 'T_mature', 'T_αβT(entry)', 'Mac', 'cTEC', 'mcTEC', 'mTEC', 'TEC-EMT', 'Fb', 'EC-blood', 'B_plasma']
CMAP_HEATMAP = sns.blend_palette([THY['blue'], THY['purple'], THY['magenta'], THY['orange'], THY['yellow']], as_cmap=True)


def mm(width, height):
    """figure size in inches from millimetres"""
    return (width / 25.4, height / 25.4)


FIG_SIZES = []


def save_exact(fig, name, png=False):
    """Save at the exact canvas size (the house style's savefig.bbox='tight' would re-crop it)."""
    path = f'{PLOTS}/{PREFIX}_{name}.pdf'
    with plt.rc_context({'savefig.bbox': 'standard'}):
        fig.savefig(path)
        if png:
            fig.savefig(path.replace('.pdf', '.png'), dpi=300, facecolor='white', transparent=False)
    w, h = np.asarray(fig.get_size_inches()) * 25.4
    FIG_SIZES.append({'figure': os.path.basename(path), 'width_mm': round(float(w), 1), 'height_mm': round(float(h), 1)})
    print(f'saved {path}  ({w:.0f} x {h:.0f} mm)')


print(f'{today} | decoupler {dc.__version__} | anndata {ad.__version__} | numpy {np.__version__}')

2026-09-07 | decoupler 2.1.1 | anndata 0.11.4 | numpy 2.2.4


## 2 · The signature: mcTEC adult-vs-infant DE genes → decoupler network

One source, `mcTEC_ageing`; targets = the mcTEC genes with `padj < 0.05`, `baseMean ≥ 10` and
`|log2FC| ≥ 0.5` in the scRNA-seq adult-vs-infant contrast; weight = the signed log2 fold change, so
**+ = up with age**. Only genes of the **scoring panel** are kept: the 4,993 genes present both on the
5,001-gene v1 panel and in the cell object. The two infant sections (U48, U51) were run on a later panel
with 50 extra genes; those are confounded with age and are excluded from every score.

In [2]:
with h5py.File(H5AD_LABELS, 'r') as f:
    ALL_GENES = pd.Index(np.asarray(read_elem(f['var'][f['var'].attrs['_index']]), dtype=str))
with h5py.File(PANEL_H5AD, 'r') as f:
    pb_var = read_elem(f['var'])
SCORING_GENES = pd.Index(np.asarray(pb_var.index[pb_var['shared_panel'].to_numpy(bool)], dtype=str))
assert SCORING_GENES.isin(ALL_GENES).all()
print(f'{len(ALL_GENES):,} genes in the cell object | {len(SCORING_GENES):,} scoring genes')

de = pd.read_csv(DE_CSV)
de = de[de['cell_type'] == CELL_TYPE].dropna(subset=['gene_name', 'padj', 'log2FoldChange'])
sig = de[(de['padj'] < PADJ_MAX) & (de['baseMean'] >= BASEMEAN_MIN) & (de['log2FoldChange'].abs() >= LFC_MIN)]
on_panel = sig[sig['gene_name'].isin(SCORING_GENES)]
net = pd.DataFrame({'source': f'{CELL_TYPE}_ageing',
                    'target': on_panel['gene_name'].to_numpy(),
                    'weight': on_panel['log2FoldChange'].to_numpy()})
N_UP, N_DOWN = int((net['weight'] > 0).sum()), int((net['weight'] < 0).sum())
print(f'{CELL_TYPE} adult vs infant: {len(de):,} genes tested, {len(sig):,} pass the filter, '
      f'{len(net):,} of those are on the scoring panel ({N_UP} up with age, {N_DOWN} down with age)')
net.to_csv(f'{RESULTS}/{PREFIX}_signature_net.csv', index=False)

w = net.set_index('target')['weight']
print('strongest UP  :', ', '.join(f'{g} ({v:+.1f})' for g, v in w.sort_values(ascending=False).head(12).items()))
print('strongest DOWN:', ', '.join(f'{g} ({v:+.1f})' for g, v in w.sort_values().head(12).items()))

5,043 genes in the cell object | 4,993 scoring genes


mcTEC adult vs infant: 16,658 genes tested, 1,558 pass the filter, 446 of those are on the scoring panel (192 up with age, 254 down with age)
strongest UP  : AKR1C3 (+5.7), PLIN5 (+4.3), SERPINA3 (+3.7), ARG2 (+3.6), DIRAS3 (+3.5), ZBTB16 (+3.1), IL33 (+3.1), CMTM8 (+3.0), CYP3A5 (+3.0), ASCL1 (+3.0), CXCL6 (+2.9), MLXIPL (+2.9)
strongest DOWN: CCL25 (-6.2), PRSS16 (-5.7), IRF8 (-4.6), GJB6 (-3.8), VCAM1 (-3.6), LY75 (-3.4), ROR1 (-3.4), FOXN1 (-3.3), IFI44L (-3.2), TUBB2B (-3.1), NTRK2 (-3.0), IFIT3 (-3.0)


## 3 · Load the mcTEC cells

Only the mcTEC rows of the 18 GB object are read, streamed in contiguous chunks from the on-disk CSR
matrix (`anndata 0.11` cannot open this file whole because of a `None` in `uns`, and contiguous chunk
reads are ~170× faster than fancy indexing). Coordinates (µm, per section) and per-cell transcript
totals come from the Proseg object, which holds the same cells in the same order (asserted, not
assumed). All-cell coordinates are kept for the grey background of the spatial maps.

In [3]:
def read_rows(dset, mask, chunk=250_000):
    """Rows `mask` of an on-disk CSR matrix, read as contiguous chunks and filtered in memory."""
    blocks = []
    for start in range(0, len(mask), chunk):
        m = mask[start:start + chunk]
        if m.any():
            blocks.append(dset[start:start + chunk][m])
    return sp.vstack(blocks, format='csr')


t0 = time.time()
with h5py.File(H5AD_LABELS, 'r') as f:
    obs_names = pd.Index(np.asarray(read_elem(f['obs'][f['obs'].attrs['_index']]), dtype=str))
    donor_all = pd.Series(np.asarray(read_elem(f['obs/donor'])).astype(str), index=obs_names)
    anno = pd.read_csv(ANNO_CSV, usecols=['proseg_barcode', CT_KEY], index_col='proseg_barcode')
    ct_all = anno[CT_KEY].reindex(obs_names)
    is_mctec = ((ct_all == CELL_TYPE) & (donor_all != 'A45')).to_numpy()
    print(f'{len(obs_names):,} cells in the object; {is_mctec.sum():,} {CELL_TYPE} in the seven donors (A45 excluded)')
    X = read_rows(sparse_dataset(f['X']), is_mctec)
print(f'expression read in {time.time() - t0:.0f} s: {X.shape[0]:,} cells x {X.shape[1]:,} genes, {X.nnz:,} non-zeros')

with h5py.File(H5AD_PROSEG, 'r') as f:
    proseg_names = np.asarray(read_elem(f['obs'][f['obs'].attrs['_index']]), dtype=str)
    assert np.array_equal(np.char.add('xenium_', proseg_names), obs_names.to_numpy()), 'cell order differs between the two objects'
    xy_all = np.asarray(read_elem(f['obsm/spatial']), dtype=np.float32)
    total_counts_all = np.asarray(read_elem(f['obs/total_counts']), dtype=np.float64)

obs = pd.DataFrame({'donor': donor_all[is_mctec].to_numpy(),
                    'x': xy_all[is_mctec, 0], 'y': xy_all[is_mctec, 1],
                    'total_counts': total_counts_all[is_mctec]}, index=obs_names[is_mctec])
obs['donor'] = pd.Categorical(obs['donor'], categories=DONOR_ORDER)
obs['age_group'] = pd.Categorical(obs['donor'].map(DONOR_AGE), categories=AGE_ORDER)
obs['age_years'] = obs['donor'].map(DONOR_AGE_YEARS).astype(float)

# what is in .X: back-transforming should give ~ a constant library size if it is log1p(counts per 10k)
back = np.asarray(X.expm1().sum(axis=1)).ravel()
print(f'.X back-transformed row sums: median {np.median(back):,.0f} (IQR {np.percentile(back, 25):,.0f}-{np.percentile(back, 75):,.0f}) '
      f'-> .X is log1p of counts normalised to 10,000 per cell')

adata = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=ALL_GENES))
adata = adata[:, SCORING_GENES].copy()                          # scoring universe only
del X, anno, ct_all
print(adata)
adata.obs.groupby('donor', observed=True).size().rename('n_mcTEC').to_frame().T

3,531,679 cells in the object; 135,914 mcTEC in the seven donors (A45 excluded)


expression read in 29 s: 135,914 cells x 5,043 genes, 33,780,711 non-zeros


.X back-transformed row sums: median 10,000 (IQR 10,000-10,000) -> .X is log1p of counts normalised to 10,000 per cell


AnnData object with n_obs × n_vars = 135914 × 4993
    obs: 'donor', 'x', 'y', 'total_counts', 'age_group', 'age_years'


donor,U51,U48,Z16,Z17,A67,A72,A70
n_mcTEC,5756,8336,2973,11428,35111,53106,19204


## 4 · Score every cell — real network and 100 permuted networks in one pass

`decoupler`'s ULM fits, for each cell, a linear regression of the cell's log-normalised expression
over **all** scoring genes on the signature weight vector (zero for genes outside the signature); the
score is the *t*-statistic of the slope. Positive = the cell's profile resembles adult mcTEC, negative =
infant mcTEC, 0 = no resemblance either way.

Each permuted network keeps the exact weights and draws the same number of targets at random from the
scoring panel, so it is degree- and weight-matched. ULM is univariate, so scoring 101 sources together
is identical to scoring them one by one.

In [4]:
def permuted_nets(net, genes, n_perm, seed=0):
    """Degree- and weight-matched null networks: same weights, targets redrawn at random from `genes`."""
    rng = np.random.default_rng(seed)
    w = net['weight'].to_numpy()
    return pd.concat([pd.DataFrame({'source': f'perm_{i:03d}', 'target': rng.choice(genes, size=len(w), replace=False), 'weight': w})
                      for i in range(n_perm)], ignore_index=True)


SOURCE = f'{CELL_TYPE}_ageing'
PERM_SOURCES = [f'perm_{i:03d}' for i in range(N_PERM)]
net_all = pd.concat([net, permuted_nets(net, SCORING_GENES.to_numpy(), N_PERM, SEED)], ignore_index=True)

t0 = time.time()
dc.mt.ulm(data=adata, net=net_all, tmin=TMIN, bsize=25_000, verbose=False)
scores = pd.DataFrame(adata.obsm['score_ulm'])
scores.index = adata.obs_names
assert SOURCE in scores.columns and all(s in scores.columns for s in PERM_SOURCES), 'a source was pruned by tmin'
print(f'ULM: {scores.shape[0]:,} cells x {scores.shape[1]} sources in {time.time() - t0:.0f} s')

perm = scores[PERM_SOURCES]                                    # cells x N_PERM
adata.obs['age_score'] = scores[SOURCE].to_numpy()
adata.obs['perm_mean'] = perm.mean(axis=1).to_numpy()
adata.obs['perm_sd'] = perm.std(axis=1).to_numpy()

# the null distribution of the per-cell score, pooled over cells and permutations
NULL_HI = float(np.quantile(perm.to_numpy().ravel(), NULL_Q))
NULL_LO = float(np.quantile(perm.to_numpy().ravel(), 1 - NULL_Q))
SD_OBS, SD_NULL = float(adata.obs['age_score'].std()), float(perm.std(axis=0).mean())
print(f'per-cell score: observed SD {SD_OBS:.2f} vs permuted-network SD {SD_NULL:.2f} (ratio {SD_OBS / SD_NULL:.2f})')
print(f'permuted-network score quantiles: {100 * (1 - NULL_Q):.0f}% = {NULL_LO:+.2f}, 50% = {np.median(perm.to_numpy()):+.2f}, {100 * NULL_Q:.0f}% = {NULL_HI:+.2f}')
print(f'observed cells above the {100 * NULL_Q:.0f}% null quantile: {100 * (adata.obs["age_score"] > NULL_HI).mean():.1f}% '
      f'(expected {100 * (1 - NULL_Q):.0f}%); below the {100 * (1 - NULL_Q):.0f}% quantile: {100 * (adata.obs["age_score"] < NULL_LO).mean():.1f}%')
adata.obs[['age_score', 'perm_mean', 'perm_sd']].describe().round(2)

ULM: 135,914 cells x 101 sources in 16 s


per-cell score: observed SD 2.21 vs permuted-network SD 0.95 (ratio 2.33)
permuted-network score quantiles: 5% = -1.63, 50% = +0.04, 95% = +1.65
observed cells above the 95% null quantile: 44.1% (expected 5%); below the 5% quantile: 9.4%


,age_score,perm_mean,perm_sd
count,135914.00,135914.00,135914.00
mean,1.19,0.02,1.00
std,2.21,0.09,0.07
min,-11.39,-0.45,0.69
25%,0.01,-0.04,0.94
50%,1.35,0.02,0.99
75%,2.66,0.09,1.04
max,8.69,0.46,1.37


## 5 · Control: does the score track the age of the section?

One section per donor, so the section mean is the natural unit. Three summaries, each computed for the
real network and for every permuted network: Spearman ρ between the section-mean score and donor age
(years); the difference of means between the three adult and the four young sections; and the per-cell
AUROC of adult vs young cells (0.5 = no separation). The empirical p-value is the share of permuted
networks that do at least as well. A per-donor table also reports how strongly the score tracks the
cell's transcript count (depth), which per-cell ULM scores are known to do in sparse data.

In [5]:
ages = pd.Series(DONOR_AGE_YEARS).loc[DONOR_ORDER]
by_donor = scores.groupby(adata.obs['donor'], observed=True).mean().loc[DONOR_ORDER]     # donors x (1 + N_PERM) mean scores
is_adult = (adata.obs['age_group'] == 'adult').to_numpy()


def age_statistics(col):
    return {'rho_age': float(spearmanr(by_donor[col], ages).statistic),
            'delta_adult_vs_young': float(by_donor.loc[ADULT, col].mean() - by_donor.loc[YOUNG, col].mean()),
            'auroc_adult_vs_young': float(roc_auc_score(is_adult, scores[col].to_numpy()))}


age_stats = pd.DataFrame([age_statistics(c) for c in scores.columns], index=scores.columns)
age_stats['network'] = np.where(age_stats.index == SOURCE, 'observed', 'permuted')
obs_stat, null_stat = age_stats.loc[SOURCE], age_stats.loc[PERM_SOURCES]
P_RHO = float((1 + (null_stat['rho_age'].abs() >= abs(obs_stat['rho_age'])).sum()) / (N_PERM + 1))
P_DELTA = float((1 + (null_stat['delta_adult_vs_young'].abs() >= abs(obs_stat['delta_adult_vs_young'])).sum()) / (N_PERM + 1))
P_AUROC = float((1 + ((null_stat['auroc_adult_vs_young'] - 0.5).abs() >= abs(obs_stat['auroc_adult_vs_young'] - 0.5)).sum()) / (N_PERM + 1))
age_stats.to_csv(f'{RESULTS}/{PREFIX}_age_control_all_networks.csv')

print(f'section-mean score vs donor age: Spearman rho = {obs_stat["rho_age"]:.3f}  '
      f'(permuted networks: median {null_stat["rho_age"].median():+.2f}, range {null_stat["rho_age"].min():+.2f}..{null_stat["rho_age"].max():+.2f}; p = {P_RHO:.3f})')
print(f'adult - young section means:     {obs_stat["delta_adult_vs_young"]:+.3f}  '
      f'(permuted: {null_stat["delta_adult_vs_young"].abs().max():.3f} at most; p = {P_DELTA:.3f})')
print(f'per-cell AUROC adult vs young:   {obs_stat["auroc_adult_vs_young"]:.3f}  '
      f'(permuted: {null_stat["auroc_adult_vs_young"].min():.3f}..{null_stat["auroc_adult_vs_young"].max():.3f}; p = {P_AUROC:.3f})')

# per-donor QC: mean, spread vs null spread, coupling to depth
rows = []
for d, g in adata.obs.groupby('donor', observed=True):
    p = perm.loc[g.index]
    rows.append({'donor': d, 'age_group': DONOR_AGE[d], 'age_years': DONOR_AGE_YEARS[d], 'n_cells': len(g),
                 'mean_score': g['age_score'].mean(), 'mean_score_null': float(p.to_numpy().mean()),
                 'sd_score': g['age_score'].std(), 'sd_score_null': float(p.std(axis=0).mean()),
                 'rho_score_depth': float(spearmanr(g['age_score'], np.log1p(g['total_counts'])).statistic),
                 'rho_score_depth_null': float(np.mean([spearmanr(p[c], np.log1p(g['total_counts'])).statistic for c in PERM_SOURCES[:20]]))})
donor_qc = pd.DataFrame(rows).set_index('donor').loc[DONOR_ORDER]
donor_qc.to_csv(f'{RESULTS}/{PREFIX}_per_donor_qc.csv')
donor_qc.round(3)

section-mean score vs donor age: Spearman rho = 0.964  (permuted networks: median +0.00, range -0.96..+1.00; p = 0.050)
adult - young section means:     +3.693  (permuted: 0.412 at most; p = 0.010)
per-cell AUROC adult vs young:   0.932  (permuted: 0.383..0.582; p = 0.010)


,age_group,age_years,n_cells,mean_score,mean_score_null,sd_score,sd_score_null,rho_score_depth,rho_score_depth_null
donor,,,,,,,,,
U51,epaed,0.04,5756,-2.417,0.003,2.179,0.972,-0.416,-0.004
U48,epaed,0.50,8336,-2.085,0.028,2.030,0.963,-0.312,-0.011
Z16,lpaed,8.00,2973,-1.614,0.011,1.947,0.963,-0.431,-0.012
Z17,lpaed,11.00,11428,-0.824,0.013,1.665,0.951,-0.209,-0.009
A67,adult,20.00,35111,1.994,0.020,1.607,0.953,0.418,-0.007
A72,adult,33.00,53106,1.837,0.034,1.578,0.933,0.401,-0.005
A70,adult,37.00,19204,2.042,0.019,1.498,0.947,0.534,0.001


In [6]:
fig, axes = plt.subplots(1, 3, figsize=mm(130, 42), layout='constrained', gridspec_kw={'width_ratios': [1.4, 1, 1]})

ax = axes[0]                                                    # section means vs age, real vs permuted
for c in PERM_SOURCES:
    ax.plot(ages.to_numpy(), by_donor[c].to_numpy(), color=GREY['grey2'], lw=0.3, zorder=1)
ax.plot(ages.to_numpy(), by_donor[SOURCE].to_numpy(), color='k', lw=0.6, zorder=2)
ax.scatter(ages.to_numpy(), by_donor[SOURCE].to_numpy(), c=[AGE_COLORS[DONOR_AGE[d]] for d in DONOR_ORDER], s=14, zorder=3, edgecolor='k', linewidth=0.3)
for d in DONOR_ORDER:
    ax.annotate(d, (ages[d], by_donor.loc[d, SOURCE]), xytext=(2, 3), textcoords='offset points', fontsize=4)
ax.axhline(0, color=GREY['grey4'], lw=0.5, ls=':')
ax.set_xscale('log')
ax.minorticks_off()
ax.set_xticks([0.04, 0.5, 8, 20, 37])
ax.set_xticklabels(['2 w', '6 m', '8 y', '20 y', '37 y'])
ax.set_xlabel('donor age')
ax.set_ylabel('mean mcTEC ageing score per section')
ax.set_title(f'real network (black) vs {N_PERM} permuted networks (grey)\nSpearman ρ = {obs_stat["rho_age"]:.2f}, p = {P_RHO:.2f}')

for ax, col, obs_v, p, lab in [(axes[1], 'rho_age', obs_stat['rho_age'], P_RHO, 'Spearman ρ (section mean vs age)'),
                               (axes[2], 'auroc_adult_vs_young', obs_stat['auroc_adult_vs_young'], P_AUROC, 'per-cell AUROC (adult vs young)')]:
    ax.hist(null_stat[col], bins=25, color=GREY['grey2'], edgecolor='none')
    ax.axvline(obs_v, color=THY['orange'], lw=1)
    ax.set_xlabel(lab)
    ax.set_ylabel('permuted networks')
    ax.set_title(f'observed {obs_v:.2f}, p = {p:.2f}')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
save_exact(fig, 'age_control')
plt.show()

saved /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/08_mcTEC_ageDE/xenium_mcTEC_ageDE_age_control.pdf  (130 x 42 mm)


## 6 · Aged-like mcTEC per donor

Two definitions, both anchored on 0:

- **aged-like** — score > 0: the cell's profile resembles adult more than infant mcTEC. Under a
  permuted network about half of the cells fall on either side, which is the reference shown in grey.
- **aged-like (strict)** — score above the 95th percentile of all permuted-network scores, i.e. a
  resemblance stronger than a random gene set of the same size and weights gives to 95 % of cells.
  The expectation under the null is 5 % by construction; it is recomputed per donor because the
  null score spread varies with depth.

In [7]:
adata.obs['aged_like'] = adata.obs['age_score'] > 0
adata.obs['aged_like_strict'] = adata.obs['age_score'] > NULL_HI
adata.obs['young_like_strict'] = adata.obs['age_score'] < NULL_LO

per_donor = adata.obs.groupby('donor', observed=True).agg(
    n_cells=('age_score', 'size'), age_group=('age_group', 'first'), age_years=('age_years', 'first'),
    mean_score=('age_score', 'mean'), median_score=('age_score', 'median'), sd_score=('age_score', 'std'),
    frac_aged_like=('aged_like', 'mean'), frac_aged_like_strict=('aged_like_strict', 'mean'),
    frac_young_like_strict=('young_like_strict', 'mean')).loc[DONOR_ORDER]

null_frac0 = (perm > 0).groupby(adata.obs['donor'], observed=True).mean().loc[DONOR_ORDER]          # donors x N_PERM
null_fracS = (perm > NULL_HI).groupby(adata.obs['donor'], observed=True).mean().loc[DONOR_ORDER]
per_donor['frac_aged_like_null'] = null_frac0.mean(axis=1)
per_donor['frac_aged_like_null_sd'] = null_frac0.std(axis=1)
per_donor['frac_aged_like_strict_null'] = null_fracS.mean(axis=1)
per_donor['frac_aged_like_strict_null_sd'] = null_fracS.std(axis=1)
per_donor.to_csv(f'{RESULTS}/{PREFIX}_per_donor_aged_like_fraction.csv')

RHO_FRAC = spearmanr(per_donor['frac_aged_like'], per_donor['age_years'])
RHO_FRAC_S = spearmanr(per_donor['frac_aged_like_strict'], per_donor['age_years'])
print(f'fraction aged-like vs age: Spearman rho = {RHO_FRAC.statistic:.3f} (p = {RHO_FRAC.pvalue:.3f}); '
      f'strict: rho = {RHO_FRAC_S.statistic:.3f} (p = {RHO_FRAC_S.pvalue:.3f})')
per_donor.round(3)

fraction aged-like vs age: Spearman rho = 0.964 (p = 0.000); strict: rho = 0.964 (p = 0.000)


,n_cells,age_group,age_years,mean_score,median_score,sd_score,frac_aged_like,frac_aged_like_strict,frac_young_like_strict,frac_aged_like_null,frac_aged_like_null_sd,frac_aged_like_strict_null,frac_aged_like_strict_null_sd
donor,,,,,,,,,,,,,
U51,5756,epaed,0.04,-2.417,-2.046,2.179,0.111,0.003,0.574,0.506,0.092,0.048,0.029
U48,8336,epaed,0.50,-2.085,-1.705,2.030,0.130,0.005,0.518,0.517,0.103,0.050,0.030
Z16,2973,lpaed,8.00,-1.614,-1.252,1.947,0.205,0.014,0.429,0.510,0.098,0.048,0.029
Z17,11428,lpaed,11.00,-0.824,-0.607,1.665,0.327,0.039,0.259,0.510,0.116,0.049,0.035
A67,35111,adult,20.00,1.994,1.902,1.607,0.900,0.560,0.006,0.513,0.122,0.050,0.036
A72,53106,adult,33.00,1.837,1.793,1.578,0.884,0.536,0.012,0.518,0.134,0.050,0.038
A70,19204,adult,37.00,2.042,1.997,1.498,0.919,0.588,0.003,0.512,0.127,0.050,0.040


In [8]:
fig, axes = plt.subplots(1, 2, figsize=mm(89, 40), layout='constrained')
x = np.arange(len(DONOR_ORDER))
for ax, col, null_col, ylab in [(axes[0], 'frac_aged_like', 'frac_aged_like_null', 'fraction of mcTEC with score > 0'),
                                     (axes[1], 'frac_aged_like_strict', 'frac_aged_like_strict_null',
                                      f'fraction of mcTEC above the {100 * NULL_Q:.0f}% null quantile')]:
    ax.bar(x, per_donor[col], color=[AGE_COLORS[DONOR_AGE[d]] for d in DONOR_ORDER], width=0.7, linewidth=0)
    ax.errorbar(x, per_donor[null_col], yerr=2 * per_donor[null_col + '_sd'], fmt='_', color=GREY['grey4'], ms=7, mew=0.8, capsize=0, lw=0.5)
    for xi, d in zip(x, DONOR_ORDER):
        ax.annotate(f'{100 * per_donor.loc[d, col]:.0f}%', (xi, per_donor.loc[d, col]), ha='center', va='bottom', fontsize=4, xytext=(0, 1), textcoords='offset points')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{d}\n{DONOR_AGE_LABEL[d]}' for d in DONOR_ORDER])
    ax.set_ylabel(ylab)
    ax.set_ylim(0, 1.0 if col == 'frac_aged_like' else max(0.3, 1.15 * per_donor[col].max()))
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_title(f'aged-like (score > 0); ρ(age) = {RHO_FRAC.statistic:.2f}')
axes[1].set_title(f'aged-like, strict; ρ(age) = {RHO_FRAC_S.statistic:.2f}')
handles = [plt.Rectangle((0, 0), 1, 1, fc=AGE_COLORS[a], lw=0) for a in AGE_ORDER] + [plt.Line2D([], [], color=GREY['grey4'], marker='_', ls='', ms=7)]
axes[1].legend(handles, AGE_ORDER + ['permuted networks (mean ± 2 SD)'], frameon=False, loc='upper left', fontsize=4)
save_exact(fig, 'aged_like_fraction_per_donor')
plt.show()

saved /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/08_mcTEC_ageDE/xenium_mcTEC_ageDE_aged_like_fraction_per_donor.pdf  (89 x 40 mm)


## 7 · Spatial maps

One panel per section, youngest to oldest. Every other cell is grey; mcTEC are coloured by their score
on a symmetric scale **centred on 0** (white), clipped at the 99th percentile of |score| so a few extreme
cells do not flatten the rest. Cells are drawn in random order so neither colour systematically covers
the other. Both a vector PDF (points rasterised) and a PNG are written.

In [9]:
VMAX = float(np.quantile(adata.obs['age_score'].abs(), 0.99))
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(2, 4, figsize=mm(183, 92), layout='constrained')
for ax, d in zip(axes.ravel(), DONOR_ORDER):
    bg = np.flatnonzero((donor_all == d).to_numpy())
    if len(bg) > 200_000:
        bg = rng.choice(bg, 200_000, replace=False)
    ax.scatter(xy_all[bg, 0], xy_all[bg, 1], s=0.05, c=GREY['grey2'], lw=0, rasterized=True)
    o = adata.obs[adata.obs['donor'] == d].sample(frac=1, random_state=SEED)
    sc_ = ax.scatter(o['x'], o['y'], s=0.6, c=o['age_score'], cmap=CMAP_SCORE, vmin=-VMAX, vmax=VMAX, lw=0, rasterized=True)
    ax.set_title(f'{d} · {DONOR_AGE[d]} · {DONOR_AGE_LABEL[d]} · n = {len(o):,} · {100 * o["aged_like"].mean():.0f}% aged-like')
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')
lax = axes.ravel()[-1]
lax.axis('off')
cax = lax.inset_axes([0.1, 0.55, 0.8, 0.07])
cb = fig.colorbar(sc_, cax=cax, orientation='horizontal')
cb.set_label('mcTEC ageing score (ULM t)\nyoung-like  ←  0  →  aged-like')
lax.text(0.5, 0.8, 'grey: all other cells; colour: mcTEC', ha='center', va='center', transform=lax.transAxes, fontsize=5)
save_exact(fig, 'spatial_maps_score', png=True)
plt.show()

saved /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/08_mcTEC_ageDE/xenium_mcTEC_ageDE_spatial_maps_score.pdf  (183 x 92 mm)


## 8 · The score across the six mcTEC niches

The niche labels come straight from `morphoFM/00_mcTEC_niche_VK_k6.ipynb` (50 µm neighbourhood
composition → log2 over the donor's tissue background → PCA → k-means, k = 6; mcTEC with fewer than 10
neighbours carry no label). The rows of every figure below follow the row order of that notebook's
published heatmaps, which is re-derived here from the saved enrichment profiles as a check.

Two things to keep in mind when reading the boxes: the niches are strongly age-associated (N2 is the
young cortex, N1 is almost exclusively adult), so the all-cell boxplot mixes the niche effect with the
donor effect; the donor-wise tables and figures separate the two.

In [10]:
niche = pd.read_csv(NICHE_CSV, index_col=0)[NICHE_KEY]
adata.obs['niche'] = pd.Categorical(niche.reindex(adata.obs_names), categories=HEATMAP_ROW_ORDER)
n_lab = int(adata.obs['niche'].notna().sum())
n_extra = int((~niche.index.isin(adata.obs_names)).sum())
print(f'{n_lab:,} of {adata.n_obs:,} mcTEC carry a niche label ({adata.n_obs - n_lab:,} have none); '
      f'{n_extra} niche-labelled cells are not in this mcTEC set')

# the published row order = leaves of average-linkage / correlation-distance clustering of the taa_l3 enrichment profiles
enr = pd.read_csv(NICHE_ENR_CSV, index_col=0)
link = linkage(enr.to_numpy(), method='average', metric='correlation')
dendro = dendrogram(link, no_plot=True)
order_recomputed = [enr.index[i] for i in dendro['leaves']]
ORDER_MATCHES = order_recomputed == HEATMAP_ROW_ORDER
print('published heatmap row order :', HEATMAP_ROW_ORDER)
print('re-derived from saved profiles:', order_recomputed, '-> match' if ORDER_MATCHES else '-> DIFFERENT (keeping the published order; no dendrogram drawn)')

lab = adata.obs.dropna(subset=['niche'])
pd.crosstab(lab['niche'], lab['donor']).loc[HEATMAP_ROW_ORDER, DONOR_ORDER]

132,575 of 135,914 mcTEC carry a niche label (3,339 have none); 0 niche-labelled cells are not in this mcTEC set
published heatmap row order : ['mcTEC_N0', 'mcTEC_N2', 'mcTEC_N5', 'mcTEC_N4', 'mcTEC_N1', 'mcTEC_N3']
re-derived from saved profiles: ['mcTEC_N0', 'mcTEC_N2', 'mcTEC_N5', 'mcTEC_N4', 'mcTEC_N1', 'mcTEC_N3'] -> match


donor,U51,U48,Z16,Z17,A67,A72,A70
niche,,,,,,,
mcTEC_N0,100,586,76,551,11644,8405,1170
mcTEC_N2,3294,3813,1638,5113,1512,15211,910
mcTEC_N5,190,270,118,385,5748,4835,1109
mcTEC_N4,480,688,161,819,4579,8908,2123
mcTEC_N1,15,41,11,250,6913,5535,10580
mcTEC_N3,1135,2192,902,4099,4177,9476,2813


In [11]:
# per-niche summary over all labelled mcTEC
rows = []
for n, g in lab.groupby('niche', observed=True):
    s = g['age_score']
    rows.append({'niche': n, 'label': NICHE_SHORT[n], 'description': NICHE_DESC[n], 'n_cells': len(s), 'n_donors': g['donor'].nunique(),
                 'frac_cells_from_adult_donors': float((g['age_group'] == 'adult').mean()),
                 'median': s.median(), 'q25': s.quantile(0.25), 'q75': s.quantile(0.75), 'mean': s.mean(),
                 'frac_aged_like': float((s > 0).mean()), 'frac_aged_like_strict': float((s > NULL_HI).mean()),
                 'frac_aged_like_null': float((perm.loc[g.index] > 0).to_numpy().mean())})
per_niche = pd.DataFrame(rows).set_index('niche').loc[HEATMAP_ROW_ORDER]
per_niche.to_csv(f'{RESULTS}/{PREFIX}_per_niche_summary.csv')

# per donor x niche: median score and n cells (groups below MIN_CELLS are blanked)
med = lab.pivot_table(index='niche', columns='donor', values='age_score', aggfunc='median', observed=True).loc[HEATMAP_ROW_ORDER, DONOR_ORDER]
ncell = lab.pivot_table(index='niche', columns='donor', values='age_score', aggfunc='count', observed=True).loc[HEATMAP_ROW_ORDER, DONOR_ORDER].fillna(0).astype(int)
med = med.where(ncell >= MIN_CELLS)
med_centred = med.sub(lab.groupby('donor', observed=True)['age_score'].median().loc[DONOR_ORDER], axis=1)   # niche effect within each donor
med.to_csv(f'{RESULTS}/{PREFIX}_niche_by_donor_median.csv')
med_centred.to_csv(f'{RESULTS}/{PREFIX}_niche_by_donor_median_centred.csv')
ncell.to_csv(f'{RESULTS}/{PREFIX}_niche_by_donor_n_cells.csv')

# do the niches differ within a donor?  Kruskal-Wallis over the niches present, with eta^2 as effect size
rows = []
for d, g in lab.groupby('donor', observed=True):
    groups = [g.loc[g['niche'] == n, 'age_score'].to_numpy() for n in HEATMAP_ROW_ORDER]
    groups = [x for x in groups if len(x) >= MIN_CELLS]
    if len(groups) < 2:
        continue
    H, p = kruskal(*groups)
    n_tot, k = sum(map(len, groups)), len(groups)
    rows.append({'donor': d, 'age_group': DONOR_AGE[d], 'n_cells': n_tot, 'n_niches': k, 'kruskal_H': H, 'p': p,
                 'eta2_H': max(0.0, (H - k + 1) / (n_tot - k)),
                 'top_niche': HEATMAP_ROW_ORDER[int(np.nanargmax(med_centred[d].to_numpy()))],
                 'bottom_niche': HEATMAP_ROW_ORDER[int(np.nanargmin(med_centred[d].to_numpy()))]})
niche_kw = pd.DataFrame(rows).set_index('donor').reindex(DONOR_ORDER)
niche_kw.to_csv(f'{RESULTS}/{PREFIX}_niche_kruskal_within_donor.csv')

print('per niche (all labelled mcTEC):')
display(per_niche.round(3))
print('median score per donor x niche (blank = fewer than', MIN_CELLS, 'cells):')
display(med.round(2))
print('the same, centred on each donor\'s median (the within-donor niche effect):')
display(med_centred.round(2))
print('Kruskal-Wallis across niches within each donor:')
niche_kw.round(4)

per niche (all labelled mcTEC):


,label,description,n_cells,n_donors,frac_cells_from_adult_donors,median,q25,q75,mean,frac_aged_like,frac_aged_like_strict,frac_aged_like_null
niche,,,,,,,,,,,,
mcTEC_N0,C*,"adult cortical (T_DP, mcTEC, Mac)",22532,7,0.942,1.356,0.280,2.528,1.366,0.803,0.433,0.515
mcTEC_N2,S,young cortex (T_DP-rich),31491,7,0.560,0.492,-1.699,2.273,0.146,0.569,0.338,0.515
mcTEC_N5,FV*,mcTEC-dense stromal,12655,7,0.924,2.141,0.834,3.399,2.046,0.870,0.598,0.517
mcTEC_N4,M,"medullary (mTEC, B, recirculating T)",17758,7,0.879,1.417,0.370,2.514,1.426,0.819,0.446,0.513
mcTEC_N1,J*,adult B-cell / vascular / fibrotic,23345,7,0.986,2.023,0.999,3.084,2.064,0.919,0.597,0.514
mcTEC_N3,MV,perimedullary / intermediate,24794,7,0.664,1.104,-0.230,2.491,1.078,0.710,0.398,0.514


median score per donor x niche (blank = fewer than 30 cells):


donor,U51,U48,Z16,Z17,A67,A72,A70
niche,,,,,,,
mcTEC_N0,-1.62,-2.08,-1.94,-1.06,1.67,1.22,1.44
mcTEC_N2,-3.06,-2.67,-2.01,-0.99,2.38,2.01,2.10
mcTEC_N5,-2.37,-1.90,-1.68,-0.82,2.61,1.99,2.29
mcTEC_N4,-0.72,-0.70,-0.57,-0.22,1.39,1.75,1.88
mcTEC_N1,NaN,-1.79,NaN,-0.29,2.10,1.88,2.11
mcTEC_N3,-1.17,-1.10,-0.52,-0.27,1.97,1.96,2.07


the same, centred on each donor's median (the within-donor niche effect):


donor,U51,U48,Z16,Z17,A67,A72,A70
niche,,,,,,,
mcTEC_N0,0.64,-0.23,-0.66,-0.45,-0.26,-0.59,-0.60
mcTEC_N2,-0.80,-0.82,-0.72,-0.38,0.45,0.19,0.06
mcTEC_N5,-0.11,-0.05,-0.39,-0.21,0.69,0.18,0.25
mcTEC_N4,1.54,1.15,0.72,0.39,-0.54,-0.06,-0.16
mcTEC_N1,NaN,0.06,NaN,0.32,0.18,0.07,0.07
mcTEC_N3,1.09,0.75,0.76,0.34,0.05,0.14,0.03


Kruskal-Wallis across niches within each donor:


,age_group,n_cells,n_niches,kruskal_H,p,eta2_H,top_niche,bottom_niche
donor,,,,,,,,
U51,epaed,5199,5,1090.9055,0.0,0.2093,mcTEC_N4,mcTEC_N2
U48,epaed,7590,6,1181.0809,0.0,0.1551,mcTEC_N4,mcTEC_N2
Z16,lpaed,2895,5,426.2231,0.0,0.1461,mcTEC_N3,mcTEC_N2
Z17,lpaed,11217,6,777.8925,0.0,0.0689,mcTEC_N4,mcTEC_N0
A67,adult,34573,6,1691.5879,0.0,0.0488,mcTEC_N5,mcTEC_N4
A72,adult,52370,6,1476.4617,0.0,0.0281,mcTEC_N2,mcTEC_N0
A70,adult,18705,6,223.2344,0.0,0.0117,mcTEC_N5,mcTEC_N0


### 8a · Marginal boxplot, in the heatmap's row order

`niche_boxplot` draws one horizontal box per niche at the vertical positions a seaborn heatmap uses
(row *i* centred on *y = i + 0.5*, row 0 at the top), so it can share a row axis with any of the
published heatmaps. Whiskers span the 5th–95th percentiles; outliers are not drawn (tens of thousands
of cells per box). Two files:

- `niche_marginal_boxplot.pdf` — the boxplot alone, 3 mm per row like the published heatmap, for
  placing beside an existing PDF;
- `niche_heatmap_with_marginal_boxplot.pdf` — the published composition heatmap (the same 12 cell types,
  % of the neighbourhood scaled to each column's maximum, from the saved profiles) redrawn with the
  boxplot on its right, guaranteed aligned.

Boxes use the figure's niche colours and short labels (`NICHE_COLORS`, `NICHE_SHORT` in section 1),
and each file is also written with the box faces at alpha 0.7 (`*_alpha07.pdf`; `BOX_ALPHAS`).

In [12]:
def niche_boxplot(ax, obs, order, key='age_score', box_h=0.62, labels='right', alpha=1.0):
    """One horizontal box per niche, rows at y = 0.5, 1.5, ... from the top (the seaborn heatmap convention).

    Boxes are filled with the niche colour at face `alpha`; tick labels are the figure's short niche names in
    the same colour (labels='right' | 'left' | None).
    """
    data = [obs.loc[obs['niche'] == n, key].to_numpy() for n in order]
    bp = ax.boxplot(data, orientation='horizontal', positions=np.arange(len(order)) + 0.5, widths=box_h, whis=(5, 95),
                    showfliers=False, patch_artist=True, manage_ticks=False)
    for patch, n in zip(bp['boxes'], order):
        patch.set_facecolor(to_rgba(NICHE_COLORS[n], alpha))
        patch.set_edgecolor('k')
        patch.set_linewidth(0.5)
    for part in ('whiskers', 'caps', 'medians'):
        for ln in bp[part]:
            ln.set_color('k')
            ln.set_linewidth(0.5)
            ln.set_linestyle('-')
    ax.axvline(0, color=GREY['grey4'], lw=0.5, ls=':', zorder=0)
    ax.set_ylim(len(order), 0)
    ax.set_yticks(np.arange(len(order)) + 0.5)
    if labels is None:
        ax.set_yticklabels([])
    else:
        if labels == 'right':
            ax.yaxis.tick_right()
        ax.set_yticklabels([NICHE_SHORT[n] for n in order], fontweight='bold')
        for tick, n in zip(ax.get_yticklabels(), order):
            tick.set_color(NICHE_COLORS[n])
    ax.tick_params(axis='y', length=0)
    ax.set_xlabel('mcTEC ageing score')
    ax.spines[['top', 'right', 'left']].set_visible(False)
    return bp


# (1) standalone: heatmap rows are 3 mm tall -> axes height = 6 x 3 mm; total canvas 34 x 32 mm
ROW_MM, N_ROWS = 3.0, len(HEATMAP_ROW_ORDER)
XLIM = (np.floor(lab['age_score'].quantile(0.02)), np.ceil(lab['age_score'].quantile(0.98)))
W_MM, H_MM = 34.0, N_ROWS * ROW_MM + 14.0
for alpha, suffix in BOX_ALPHAS:
    fig = plt.figure(figsize=mm(W_MM, H_MM))
    ax = fig.add_axes([2.0 / W_MM, 10.0 / H_MM, 18.0 / W_MM, N_ROWS * ROW_MM / H_MM])   # [left, bottom, width, height] in canvas fractions
    niche_boxplot(ax, lab, HEATMAP_ROW_ORDER, alpha=alpha)
    ax.set_xlim(*XLIM)
    save_exact(fig, f'niche_marginal_boxplot{suffix}')
    plt.show()

saved /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/08_mcTEC_ageDE/xenium_mcTEC_ageDE_niche_marginal_boxplot.pdf  (34 x 32 mm)


saved /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/08_mcTEC_ageDE/xenium_mcTEC_ageDE_niche_marginal_boxplot_alpha07.pdf  (34 x 32 mm)


In [13]:
# (2) the published composition heatmap redrawn from the saved profiles, with the boxplot on its right
pct = pd.read_csv(NICHE_PCT_CSV, index_col=0).loc[HEATMAP_ROW_ORDER, HEATMAP_COLS]
scaled = pct / pct.max(axis=0)                              # each cell type scaled to its maximum over the niches (as in the k6 notebook)


def heatmap_with_boxplot(alpha, suffix):
    """Dendrogram | composition heatmap | marginal boxplot, sharing one row axis (3 mm per row); one file per alpha variant."""
    COL_MM = 3.0
    DN_MM, HM_MM, GAP_MM, BX_MM = 6.0, COL_MM * len(HEATMAP_COLS), 3.0, 26.0
    LEFT_MM, RIGHT_MM, TOP_MM, BOTTOM_MM = 1.5, 14.0, 6.0, 14.0
    W_MM = LEFT_MM + DN_MM + HM_MM + GAP_MM + BX_MM + RIGHT_MM
    H_MM = BOTTOM_MM + N_ROWS * ROW_MM + TOP_MM
    fig = plt.figure(figsize=mm(W_MM, H_MM))
    y0, hh = BOTTOM_MM / H_MM, N_ROWS * ROW_MM / H_MM
    ax_dn = fig.add_axes([LEFT_MM / W_MM, y0, DN_MM / W_MM, hh])
    ax_hm = fig.add_axes([(LEFT_MM + DN_MM) / W_MM, y0, HM_MM / W_MM, hh])
    ax_bx = fig.add_axes([(LEFT_MM + DN_MM + HM_MM + GAP_MM) / W_MM, y0, BX_MM / W_MM, hh])
    cax = fig.add_axes([(LEFT_MM + DN_MM) / W_MM, y0 + hh + 1.5 / H_MM, 14.0 / W_MM, 1.6 / H_MM])

    sns.heatmap(scaled, ax=ax_hm, cmap=CMAP_HEATMAP, robust=True, cbar=True, cbar_ax=cax,
                cbar_kws={'orientation': 'horizontal'}, xticklabels=True, yticklabels=False, linewidths=0)
    cax.set_title('% of neighbourhood / column max', fontsize=4, pad=1)
    cax.tick_params(labelsize=4, length=1.5, pad=1)
    ax_hm.set_xticklabels(HEATMAP_COLS, rotation=90)
    ax_hm.set_xlabel('')
    ax_hm.set_ylabel('')
    ax_hm.tick_params(axis='x', length=0)

    if ORDER_MATCHES:                                         # leaf i of the tree sits at y = 10 i + 5, i.e. on heatmap row i
        for xs, ys in zip(dendro['dcoord'], dendro['icoord']):
            ax_dn.plot(xs, ys, color='k', lw=0.5)
        ax_dn.set_ylim(N_ROWS * 10, 0)
        ax_dn.set_xlim(max(max(x) for x in dendro['dcoord']) * 1.05, 0)
    ax_dn.axis('off')

    niche_boxplot(ax_bx, lab, HEATMAP_ROW_ORDER, alpha=alpha)
    ax_bx.set_xlim(*XLIM)
    save_exact(fig, f'niche_heatmap_with_marginal_boxplot{suffix}')
    return fig


for alpha, suffix in BOX_ALPHAS:
    heatmap_with_boxplot(alpha, suffix)
    plt.show()

saved /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/08_mcTEC_ageDE/xenium_mcTEC_ageDE_niche_heatmap_with_marginal_boxplot.pdf  (86 x 38 mm)


saved /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/08_mcTEC_ageDE/xenium_mcTEC_ageDE_niche_heatmap_with_marginal_boxplot_alpha07.pdf  (86 x 38 mm)


### 8b · Separating the niche effect from the donor effect

Left: median score per donor × niche (blank = fewer than 30 cells). Middle: the same after subtracting
each donor's overall median, i.e. the niche gradient *inside* a section. Right: the boxplot of 8a split
by age group.

In [14]:
fig, axes = plt.subplots(1, 3, figsize=mm(183, 44), layout='constrained', gridspec_kw={'width_ratios': [1.15, 1.15, 1.3]})
vmax = float(np.nanmax(np.abs(med.to_numpy())))
vmax_c = float(np.nanmax(np.abs(med_centred.to_numpy())))
for ax, M, v, ttl, lab_cb in [(axes[0], med, vmax, 'median score per donor × niche', 'median score'),
                              (axes[1], med_centred, vmax_c, 'centred on each donor\'s median', 'median − donor median')]:
    sns.heatmap(M, ax=ax, cmap=CMAP_SCORE, center=0, vmin=-v, vmax=v, annot=True, fmt='.1f', annot_kws={'fontsize': 4},
                linewidths=0.3, linecolor='white', cbar_kws={'label': lab_cb, 'pad': 0.02, 'aspect': 12},
                xticklabels=[f'{d}\n{DONOR_AGE_LABEL[d]}' for d in DONOR_ORDER], yticklabels=HEATMAP_ROW_ORDER)
    ax.set_title(ttl)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='both', length=0, rotation=0)

ax = axes[2]
for j, a in enumerate(AGE_ORDER):
    sub = lab[lab['age_group'] == a]
    data, pos = [], []
    for i, n in enumerate(HEATMAP_ROW_ORDER):
        v = sub.loc[sub['niche'] == n, 'age_score'].to_numpy()
        if len(v) >= MIN_CELLS:
            data.append(v)
            pos.append(i + 0.5 + (j - 1) * 0.27)
    if not data:
        continue
    bp = ax.boxplot(data, orientation='horizontal', positions=pos, widths=0.24, whis=(5, 95), showfliers=False, patch_artist=True, manage_ticks=False)
    for patch in bp['boxes']:
        patch.set_facecolor(AGE_COLORS[a])
        patch.set_edgecolor('k')
        patch.set_linewidth(0.4)
    for part in ('whiskers', 'caps', 'medians'):
        for ln in bp[part]:
            ln.set_color('k')
            ln.set_linewidth(0.4)
            ln.set_linestyle('-')
ax.axvline(0, color=GREY['grey4'], lw=0.5, ls=':', zorder=0)
ax.set_ylim(N_ROWS, 0)
ax.set_yticks(np.arange(N_ROWS) + 0.5)
ax.set_yticklabels(HEATMAP_ROW_ORDER)
ax.tick_params(axis='y', length=0)
ax.set_xlabel('mcTEC ageing score')
ax.set_title(f'by age group (≥ {MIN_CELLS} cells per box; whiskers 5–95%)')
ax.legend([plt.Rectangle((0, 0), 1, 1, fc=AGE_COLORS[a], lw=0) for a in AGE_ORDER], AGE_ORDER, frameon=False, loc='upper left', fontsize=4)
ax.spines[['top', 'right']].set_visible(False)
save_exact(fig, 'niche_score_by_donor_and_age_group')
plt.show()

saved /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/08_mcTEC_ageDE/xenium_mcTEC_ageDE_niche_score_by_donor_and_age_group.pdf  (183 x 44 mm)


### 8c · Which niches differ?

One procedure, with the **donor as the replicate**: the observation is the median score of each niche
in each donor (niches with ≥ 30 cells). A Friedman test asks whether the niches differ at all (over the
niches present in all seven donors; Kendall's W = how much the donors agree on the ordering), and a
paired Wilcoxon signed-rank test per pair of niches, Benjamini–Hochberg-corrected over the 15 pairs,
says which pairs differ. Pairing on donor removes the donor/age confound. Tests on the pooled cells
are deliberately not used: with 10⁴ cells per niche every difference is "significant", and the niches
are confounded with donor. Resolution: with seven donors the exact two-sided p cannot fall below
2/2⁷ = 0.016 (0.063 for pairs involving N1, which has ≥ 30 cells in only five donors), so the number
of donors with A > B is reported next to p.

In [15]:
PAIRS = list(combinations(HEATMAP_ROW_ORDER, 2))

# overall: Friedman test over the niches present in every donor
complete = [n for n in HEATMAP_ROW_ORDER if med.loc[n].notna().all()]
FRIED = friedmanchisquare(*[med.loc[n].to_numpy() for n in complete])
KENDALL_W = float(FRIED.statistic / (len(DONOR_ORDER) * (len(complete) - 1)))
mean_rank = med.rank(axis=0).mean(axis=1)                    # mean rank of each niche across donors (1 = lowest score)

# pairwise: paired Wilcoxon signed-rank test on the per-donor medians
rows = []
for a, b in PAIRS:
    x, y = med.loc[a], med.loc[b]
    ok = x.notna() & y.notna()
    if ok.sum() < 3:
        continue
    diff = x[ok] - y[ok]
    rows.append({'niche_a': a, 'niche_b': b, 'n_donors': int(ok.sum()),
                 'mean_diff_of_medians': float(diff.mean()), 'n_donors_a_higher': int((diff > 0).sum()),
                 'p_wilcoxon': float(wilcoxon(x[ok], y[ok], alternative='two-sided', method='exact').pvalue),
                 'min_p_possible': 2 / 2 ** int(ok.sum())})
niche_tests = pd.DataFrame(rows)
niche_tests['padj'] = multipletests(niche_tests['p_wilcoxon'], method='fdr_bh')[1]
niche_tests['significant'] = niche_tests['padj'] < 0.05
niche_tests.to_csv(f'{RESULTS}/{PREFIX}_niche_pairwise_tests.csv', index=False)
pd.DataFrame([{'test': 'Friedman over per-donor medians', 'niches': ','.join(complete), 'n_donors': len(DONOR_ORDER),
               'chi2': float(FRIED.statistic), 'p': float(FRIED.pvalue), 'kendall_w': KENDALL_W}]).to_csv(f'{RESULTS}/{PREFIX}_niche_friedman.csv', index=False)
SIG_PAIRS = [f'{r.niche_a} vs {r.niche_b}' for r in niche_tests.itertuples() if r.significant]

print(f'Friedman over the {len(complete)} niches present in all 7 donors ({", ".join(complete)}): '
      f'chi2 = {FRIED.statistic:.1f}, p = {FRIED.pvalue:.4f}, Kendall W = {KENDALL_W:.2f}')
print('mean rank across donors (1 = lowest score): ' + ', '.join(f'{n} {mean_rank[n]:.1f}' for n in mean_rank.sort_values().index))
print(f'pairs significant after BH: {", ".join(SIG_PAIRS) if SIG_PAIRS else "none"}')
niche_tests.round(4)

Friedman over the 5 niches present in all 7 donors (mcTEC_N0, mcTEC_N2, mcTEC_N5, mcTEC_N4, mcTEC_N3): chi2 = 8.0, p = 0.0916, Kendall W = 0.29
mean rank across donors (1 = lowest score): mcTEC_N0 1.7, mcTEC_N2 2.9, mcTEC_N4 3.7, mcTEC_N5 4.0, mcTEC_N1 4.0, mcTEC_N3 4.1
pairs significant after BH: none


,niche_a,niche_b,n_donors,mean_diff_of_medians,n_donors_a_higher,p_wilcoxon,min_p_possible,padj,significant
0,mcTEC_N0,mcTEC_N2,7,-0.0206,3,0.6875,0.0156,0.8125,False
1,mcTEC_N0,mcTEC_N5,7,-0.3608,1,0.1094,0.0156,0.3281,False
2,mcTEC_N0,mcTEC_N4,7,-0.7422,1,0.0312,0.0156,0.1562,False
3,mcTEC_N0,mcTEC_N1,5,-0.5678,0,0.0625,0.0625,0.2344,False
4,mcTEC_N0,mcTEC_N3,7,-0.7616,0,0.0156,0.0156,0.1562,False
5,mcTEC_N2,mcTEC_N5,7,-0.3402,1,0.0312,0.0156,0.1562,False
6,mcTEC_N2,mcTEC_N4,7,-0.7216,3,0.2969,0.0156,0.5566,False
7,mcTEC_N2,mcTEC_N1,5,-0.2392,2,0.6250,0.0625,0.8125,False
8,mcTEC_N2,mcTEC_N3,7,-0.7410,3,0.2188,0.0156,0.5469,False
9,mcTEC_N5,mcTEC_N4,7,-0.3815,3,0.4688,0.0156,0.7812,False


In [16]:
# niche x niche: mean difference of per-donor medians (row - column); annotation = donors with row > column, stars = BH-adjusted paired Wilcoxon
def star(p):
    return '' if not np.isfinite(p) else ('***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')


M = pd.DataFrame(np.nan, index=HEATMAP_ROW_ORDER, columns=HEATMAP_ROW_ORDER)
A = pd.DataFrame('', index=HEATMAP_ROW_ORDER, columns=HEATMAP_ROW_ORDER)
for r in niche_tests.itertuples():
    M.loc[r.niche_a, r.niche_b] = r.mean_diff_of_medians
    M.loc[r.niche_b, r.niche_a] = -r.mean_diff_of_medians
    A.loc[r.niche_a, r.niche_b] = f'{r.n_donors_a_higher}/{r.n_donors}{star(r.padj)}'
    A.loc[r.niche_b, r.niche_a] = f'{r.n_donors - r.n_donors_a_higher}/{r.n_donors}{star(r.padj)}'
vmax = float(np.nanmax(np.abs(M.to_numpy())))

fig, ax = plt.subplots(figsize=mm(70, 52), layout='constrained')
sns.heatmap(M, ax=ax, cmap=CMAP_SCORE, center=0, vmin=-vmax, vmax=vmax, annot=A.to_numpy(), fmt='', annot_kws={'fontsize': 4.5},
            linewidths=0.3, linecolor='white', cbar_kws={'label': 'Δ median (row − column)', 'pad': 0.02, 'aspect': 12},
            xticklabels=HEATMAP_ROW_ORDER, yticklabels=HEATMAP_ROW_ORDER)
ax.set_title('per-donor medians: mean difference (row − column)\nannotation: donors with row > column; stars = paired Wilcoxon, BH')
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='both', length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.tick_params(axis='y', rotation=0)
save_exact(fig, 'niche_pairwise_tests')
plt.show()

saved /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/08_mcTEC_ageDE/xenium_mcTEC_ageDE_niche_pairwise_tests.pdf  (70 x 52 mm)


## 9 · Per-cell table, run summary

In [17]:
percell = adata.obs[['donor', 'age_group', 'age_years', 'x', 'y', 'total_counts', 'age_score', 'perm_mean', 'perm_sd',
                     'aged_like', 'aged_like_strict', 'young_like_strict', 'niche']].copy()
percell.to_csv(f'{RESULTS}/{PREFIX}_scores_percell.csv')
pd.DataFrame(FIG_SIZES).to_csv(f'{RESULTS}/{PREFIX}_figure_sizes.csv', index=False)

summary = {
    'date': today, 'cell_type': CELL_TYPE, 'annotation': CT_KEY,
    'signature': {'contrast': 'adult_vs_infant (taa_l3, pyDESeq2)', 'padj_max': PADJ_MAX, 'lfc_min': LFC_MIN, 'basemean_min': BASEMEAN_MIN,
                  'n_genes_tested': int(len(de)), 'n_genes_pass_filter': int(len(sig)), 'n_genes_on_scoring_panel': int(len(net)),
                  'n_up': N_UP, 'n_down': N_DOWN, 'n_scoring_genes': int(len(SCORING_GENES))},
    'cells': {'n_mcTEC': int(adata.n_obs), 'per_donor': {k: int(v) for k, v in per_donor['n_cells'].items()}, 'n_with_niche_label': n_lab},
    'null': {'n_permuted_networks': N_PERM, 'sd_observed': SD_OBS, 'sd_null': SD_NULL, 'null_quantile': NULL_Q, 'null_hi': NULL_HI, 'null_lo': NULL_LO},
    'age_control': {'rho_section_mean_vs_age': obs_stat['rho_age'], 'p_perm': P_RHO,
                    'rho_null_median': float(null_stat['rho_age'].median()), 'rho_null_max_abs': float(null_stat['rho_age'].abs().max()),
                    'delta_adult_vs_young': obs_stat['delta_adult_vs_young'], 'p_perm_delta': P_DELTA,
                    'auroc_adult_vs_young': obs_stat['auroc_adult_vs_young'], 'p_perm_auroc': P_AUROC,
                    'auroc_null_range': [float(null_stat['auroc_adult_vs_young'].min()), float(null_stat['auroc_adult_vs_young'].max())],
                    'rho_score_depth_per_donor': donor_qc['rho_score_depth'].round(3).to_dict()},
    'aged_like': {'frac_per_donor': per_donor['frac_aged_like'].round(3).to_dict(), 'frac_strict_per_donor': per_donor['frac_aged_like_strict'].round(3).to_dict(),
                  'rho_frac_vs_age': float(RHO_FRAC.statistic), 'rho_frac_strict_vs_age': float(RHO_FRAC_S.statistic)},
    'niches': {'row_order': HEATMAP_ROW_ORDER, 'row_order_recomputed_matches': bool(ORDER_MATCHES),
               'median_per_niche': per_niche['median'].round(3).to_dict(), 'frac_aged_like_per_niche': per_niche['frac_aged_like'].round(3).to_dict(),
               'frac_cells_from_adult_donors': per_niche['frac_cells_from_adult_donors'].round(3).to_dict(),
               'eta2_within_donor': niche_kw['eta2_H'].round(3).to_dict()},
    'niche_tests': {'friedman_niches': complete, 'friedman_p': float(FRIED.pvalue), 'kendall_w': KENDALL_W,
                    'mean_rank_across_donors': mean_rank.round(2).to_dict(),
                    'significant_pairs': SIG_PAIRS},
    'vmax_maps': VMAX, 'runtime_min': round((time.time() - T_START) / 60, 1),
}
with open(f'{RESULTS}/{PREFIX}_run_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2, default=str)

print(f"""
=== {PREFIX}  {today}  ({summary['runtime_min']} min) ===
signature   {len(net)} panel genes ({N_UP} up / {N_DOWN} down with age) from {len(sig):,} DE genes; scoring universe {len(SCORING_GENES):,} genes
cells       {adata.n_obs:,} mcTEC in 7 sections; {n_lab:,} with a niche label
null        {N_PERM} permuted networks; per-cell SD observed {SD_OBS:.2f} vs null {SD_NULL:.2f}; strict threshold = {NULL_HI:+.2f}
age         section mean vs age: rho = {obs_stat['rho_age']:.2f} (p = {P_RHO:.3f}); adult - young = {obs_stat['delta_adult_vs_young']:+.2f} (p = {P_DELTA:.3f});
            per-cell AUROC = {obs_stat['auroc_adult_vs_young']:.3f} (p = {P_AUROC:.3f}); |rho(score, depth)| per donor {donor_qc['rho_score_depth'].abs().min():.2f}..{donor_qc['rho_score_depth'].abs().max():.2f}
aged-like   {', '.join(f'{d} {100 * per_donor.loc[d, "frac_aged_like"]:.0f}%' for d in DONOR_ORDER)}  (rho vs age {RHO_FRAC.statistic:.2f})
niches      medians {', '.join(f'{n} {per_niche.loc[n, "median"]:+.2f}' for n in HEATMAP_ROW_ORDER)}
            within-donor eta2 {', '.join(f'{d} {niche_kw.loc[d, "eta2_H"]:.2f}' for d in DONOR_ORDER)}
tests       Friedman over per-donor medians ({len(complete)} niches x 7 donors): p = {FRIED.pvalue:.4f}, Kendall W = {KENDALL_W:.2f}
            pairs FDR < 0.05 (paired Wilcoxon on per-donor medians): {', '.join(SIG_PAIRS) if SIG_PAIRS else 'none'}
files       {RESULTS}
            {PLOTS}
""")
for f in sorted(os.listdir(RESULTS)):
    print('  ', f)
for f in sorted(os.listdir(PLOTS)):
    print('  ', f)


=== xenium_mcTEC_ageDE  2026-09-07  (1.2 min) ===
signature   446 panel genes (192 up / 254 down with age) from 1,558 DE genes; scoring universe 4,993 genes
cells       135,914 mcTEC in 7 sections; 132,575 with a niche label
null        100 permuted networks; per-cell SD observed 2.21 vs null 0.95; strict threshold = +1.65
age         section mean vs age: rho = 0.96 (p = 0.050); adult - young = +3.69 (p = 0.010);
            per-cell AUROC = 0.932 (p = 0.010); |rho(score, depth)| per donor 0.21..0.53
aged-like   U51 11%, U48 13%, Z16 21%, Z17 33%, A67 90%, A72 88%, A70 92%  (rho vs age 0.96)
niches      medians mcTEC_N0 +1.36, mcTEC_N2 +0.49, mcTEC_N5 +2.14, mcTEC_N4 +1.42, mcTEC_N1 +2.02, mcTEC_N3 +1.10
            within-donor eta2 U51 0.21, U48 0.16, Z16 0.15, Z17 0.07, A67 0.05, A72 0.03, A70 0.01
tests       Friedman over per-donor medians (5 niches x 7 donors): p = 0.0916, Kendall W = 0.29
            pairs FDR < 0.05 (paired Wilcoxon on per-donor medians): none
files       /nfs

## How to read this

1. **Section 5 is the control that matters.** The signature was built from adult-vs-infant scRNA-seq,
   so adult Xenium mcTEC scoring higher is expected; the informative comparison is with the permuted
   networks, which share the weights and the number of genes and only lose the gene identities. The
   correlation with age, the adult–young difference and the AUROC should all lie far outside the
   permuted range. Note that the permuted-network ρ distribution is *not* centred on 0: adult and
   young mcTEC differ in the expression of most genes, so the section means of any random gene set
   already separate the two groups a little and, with only seven sections, |ρ| is often high in
   either direction. The size of the effect is what the permutations cannot reproduce — read the
   adult–young difference (observed vs the largest permuted value) and the per-cell AUROC, not the
   ρ p-value alone.
2. **The per-cell score is noisy and depth-coupled** (see `rho_score_depth` in the per-donor table):
   the ULM *t*-statistic of a single Xenium cell moves with the cell's transcript count, and the sign of
   that coupling differs between young and adult sections. Reading individual cells off the maps is
   therefore not advisable; read patches of colour, and the per-donor / per-niche summaries.
3. **Niches and age are confounded**: N1 and N0 are made of adult cells, N2 of young cells, so the
   marginal boxplot largely re-states the age effect. The centred donor × niche panel and the Kruskal–
   Wallis η² say how much niche structure exists inside a section; the by-age-group boxes show whether
   the niche ordering is the same at every age.